# Get all the informations of a given SIREN

In [ ]:
pip install typing_extensions

In [ ]:
pip install webdriver_manager

In [ ]:
pip install selenium

In [ ]:
pip install seleniumbase

In [ ]:
#from webdriver_manager.chrome import ChromeDriverManager
#driver = webdriver.Chrome(ChromeDriverManager(version="107.0.5304.62").install())

# Extractions des indicateurs financiers (site Pappers)

In [ ]:
import time
import math
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# --- CONFIGURATION ---
FILE_INPUT = "bdd.xlsx"
df_siren = pd.read_excel(FILE_INPUT)
liste_siren = df_siren["SIREN"].apply(lambda x: str(int(x)) if not pd.isnull(x) else "").tolist()

options = Options()
# options.add_argument("--headless") 
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Nettoyage des valeurs pour conversion en unités réelles (Euros)
def nettoyer_valeur(valeur):
    if not valeur or valeur == "-":
        return None
    valeur = valeur.replace(",", ".").replace(" ", "").upper()
    try:
        if "MDS" in valeur or "MD" in valeur:
            return float(valeur.replace("MDS", "").replace("MD", "")) * 1_000_000_000
        elif "M" in valeur:
            return float(valeur.replace("M", "")) * 1_000_000
        elif "K" in valeur:
            return float(valeur.replace("K", "")) * 1_000
        else:
            return float(valeur)
    except:
        return None

# --- INDICATEURS FINANCIERS ---
indicateurs_utiles = [
    "chiffre d'affaires (€)",
    "résultat net (€)",
    "résultat d'exploitation (€)",
    "fonds propres (€)",
    "dettes financières (€)",
    "rentabilité sur fonds propres (%)",
    "rentabilité économique (%)",
    "ratio d'endettement (gearing)",      # Ajouté
    "chiffre d'affaires à l'export (€)"   # Ajouté
]

def extraire_indicateurs(siren):
    url = f"https://www.pappers.fr/entreprise/{siren}"
    driver.get(url)
    time.sleep(5)

    try:
        close_btn = driver.find_element(By.CLASS_NAME, "close-btn")
        close_btn.click()
        time.sleep(1)
    except:
        pass

    # Scroll profond pour charger tous les tableaux de ratios
    for _ in range(45):
        driver.execute_script("window.scrollBy(0, 350);")
        time.sleep(0.25)

    try:
        nom = driver.find_element(By.XPATH, "//h1").text.strip()
    except:
        nom = "Inconnu"

    lignes = []
    try:
        tableaux = driver.find_elements(By.XPATH, "//div[@class='ratios']//table")
        for tableau in tableaux:
            # Extraction des années
            headers = tableau.find_elements(By.XPATH, ".//tr[@class='tr-header']/th")[1:]
            annees = [int(a.text.strip()) for a in headers if a.text.strip().isdigit()]
            
            lignes_tableau = tableau.find_elements(By.TAG_NAME, "tr")
            for ligne in lignes_tableau:
                try:
                    th = ligne.find_element(By.TAG_NAME, "th").text.strip().lower()
                    if th in indicateurs_utiles:
                        tds = ligne.find_elements(By.TAG_NAME, "td")
                        for i, td in enumerate(tds):
                            if i < len(annees):
                                annee = annees[i]
                                valeur = nettoyer_valeur(td.text.strip())
                                lignes.append({
                                    "SIREN": siren,
                                    "Nom": nom,
                                    "Année": annee,
                                    "Indicateur": th,
                                    "Valeur": valeur if valeur is not None else "-"
                                })
                except:
                    continue
    except:
        pass

    if not lignes:
        return [], {"SIREN": siren, "Nom": nom}

    # --- CALCUL DU ROE & ACTIF NET AU CARRÉ ---
    df_temp = pd.DataFrame(lignes)
    for annee in df_temp["Année"].unique():
        subset = df_temp[df_temp["Année"] == annee]
        try:
            # Calcul ROE (si pas déjà présent sur le site)
            rn = subset.loc[subset["Indicateur"] == "résultat net (€)", "Valeur"].values[0]
            fp = subset.loc[subset["Indicateur"] == "fonds propres (€)", "Valeur"].values[0]
            
            if rn != "-" and fp != "-" and float(fp) != 0:
                # 1. ROE
                roe = (float(rn) / float(fp)) * 100
                lignes.append({
                    "SIREN": siren, "Nom": nom, "Année": annee,
                    "Indicateur": "roe calculé (%)", "Valeur": round(roe, 2)
                })
                # 2. ACTIF NET AU CARRÉ (Fonds propres au carré)
                actif_carre = float(fp)**2
                lignes.append({
                    "SIREN": siren, "Nom": nom, "Année": annee,
                    "Indicateur": "actif net au carré", "Valeur": actif_carre
                })
        except:
            continue

    return lignes, None

# --- BOUCLE PAR BATCH ---
batch_size = 100
nb_batches = math.ceil(len(liste_siren) / batch_size)

for batch_num in range(nb_batches):
    start = batch_num * batch_size
    end = start + batch_size
    batch_siren = liste_siren[start:end]

    print(f"\n Traitement du batch {batch_num+1}/{nb_batches}...")
    toutes_donnees = []
    entreprises_vides = []

    for siren in batch_siren:
        print(f" Extraction : {siren}")
        data, vide = extraire_indicateurs(siren)
        toutes_donnees += data
        if vide: entreprises_vides.append(vide)

    # Sauvegarde Batch
    if toutes_donnees:
        df_final = pd.DataFrame(toutes_donnees)
        df_final.sort_values(by=["Nom", "Année", "Indicateur"], inplace=True)
        df_final.to_excel(f"resultats_batch_{batch_num+1}.xlsx", index=False)

    if entreprises_vides:
        pd.DataFrame(entreprises_vides).to_excel(f"vides_batch_{batch_num+1}.xlsx", index=False)

print("\n Extraction terminée.")
driver.quit()

# Merge des fichiers obtenus en un seul fichier

In [ ]:
import pandas as pd
import glob

# ==============================
# 1. MERGER TOUS LES BATCHS
# ==============================

# Récupère tous les fichiers resultats_batch_*.xlsx
fichiers = glob.glob( r"resultats_batch_*.xlsx")

# Lecture et concaténation
dfs = [pd.read_excel(f) for f in fichiers]
df_merged = pd.concat(dfs, ignore_index=True)

print("Nb lignes merge :", len(df_merged))


# ==============================
# 2. PIVOT : indicateurs → colonnes
# ==============================

df_pivot = df_merged.pivot_table(
    index=["SIREN", "Nom", "Année"],
    columns="Indicateur",
    values="Valeur",
    aggfunc="first"
).reset_index()

# Nettoyage nom colonnes (optionnel)
df_pivot.columns.name = None
df_pivot.to_excel("resultats_indic_financiers.xlsx", index=False)
print("Nb lignes pivot :", len(df_pivot))




# concatenation de la base resultat avec BDD originale

In [ ]:


# ==============================
# CHARGEMENT
# ==============================

df_bdd = pd.read_excel("bdd.xlsx")

df_pivot = df_pivot.drop(columns=["Nom"], errors="ignore")


# ==============================
# FILTRER SIREN
# ==============================

df_pivot_filtre = df_pivot[
    df_pivot["SIREN"].isin(df_bdd["SIREN"])
]

# ==============================
# MERGE → PANEL
# ==============================

df_final = pd.merge(
    df_bdd,
    df_pivot_filtre,
    on="SIREN",
    how="left"
)

# ==============================
# SUPPRIMER HEURE DES DATES
# ==============================

for col in df_final.columns:
    if pd.api.types.is_datetime64_any_dtype(df_final[col]):
        df_final[col] = df_final[col].dt.date

# ==============================
# SAUVEGARDE
# ==============================

df_final.to_excel("bdd_panel.xlsx", index=False)

print("Base panel créée ✅")

## Extraction 'total actif' avec des batchs (verif.com)

In [ ]:
import time
import pandas as pd
import numpy as np
import os
import glob
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from seleniumbase import Driver

# --- CONFIGURATION ---
FILE_INPUT = "bdd_panel.xlsx"
FINAL_OUTPUT = "bdd_complet_T_actif.xlsx"
BATCH_SIZE = 100
COL_SIREN = "SIREN"
COL_ANNEE = "Année"

def convertir_valeur(txt):
    if not txt or txt == "--": return 0
    txt = txt.replace(",", ".").replace(" ", "").upper()
    try:
        if "B" in txt: return float(txt.replace("B", "")) * 1_000_000_000
        if "M" in txt: return float(txt.replace("M", "")) * 1_000_000
        if "K" in txt: return float(txt.replace("K", "")) * 1_000
        return float(txt)
    except:
        return 0

# 1. Préparation des données
print(f"Lecture de {FILE_INPUT}...")
df_global = pd.read_excel(FILE_INPUT, dtype={COL_SIREN: str})

if COL_ANNEE not in df_global.columns:
    df_global[COL_ANNEE] = np.nan

sirens_uniques = df_global[COL_SIREN].dropna().unique()
# Création des groupes de 100
batches = [sirens_uniques[i:i + BATCH_SIZE] for i in range(0, len(sirens_uniques), BATCH_SIZE)]

# 2. Boucle de traitement par Batch
driver = Driver(uc=True, headless=False)
wait = WebDriverWait(driver, 15)

try:
    for i, batch_sirens in enumerate(batches, start=1):
        batch_filename = f"bdd_batch_{i}.xlsx"
        
        # Sécurité : Si le fichier du batch existe déjà, on le passe (reprise après crash)
        if os.path.exists(batch_filename):
            print(f"Batch {i} déjà traité. Passage au suivant...")
            continue
            
        print(f"\n>>> DÉBUT DU BATCH {i}/{len(batches)} ({len(batch_sirens)} SIRENs)")
        
        # On travaille sur une copie de la portion du dataframe concernée par ce batch
        df_batch = df_global[df_global[COL_SIREN].isin(batch_sirens)].copy()
        df_batch["Total Actif"] = None

        for siren in batch_sirens:
            print(f"SIREN : {siren}", end=" | ")
            try:
                url = f"https://www.verif.com/societe/{siren}/"
                driver.uc_open_with_reconnect(url, 4)
                driver.execute_script("window.scrollTo(0, 800)")
                time.sleep(1.5)

                # Extraction Header & Données
                header_row = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'Evolution')]/parent::div")))
                annees_site = [s.text.strip() for s in header_row.find_elements(By.TAG_NAME, "span") if s.text.strip().isdigit()]
                
                ligne_actif = wait.until(EC.presence_of_element_located((By.XPATH, "//span[text()='Total Actif']/parent::div")))
                chiffres_bruts = [v.text.strip() for v in ligne_actif.find_elements(By.TAG_NAME, "span")[1:] if "%" not in v.text]

                map_data = {int(annees_site[j]): convertir_valeur(chiffres_bruts[j]) for j in range(min(len(annees_site), len(chiffres_bruts)))}

                # Mise à jour des lignes de ce SIREN dans le batch actuel
                indices = df_batch[df_batch[COL_SIREN] == siren].index
                for idx in indices:
                    val_annee = df_batch.at[idx, COL_ANNEE]
                    
                    if pd.isna(val_annee):
                        annees_valides = [y for y in map_data.keys() if 2021 <= y <= 2024]
                        if annees_valides:
                            derniere = max(annees_valides)
                            df_batch.at[idx, "Total Actif"] = map_data[derniere]
                            print(f"Auto {derniere}: {map_data[derniere]}€")
                        else:
                            df_batch.at[idx, "Total Actif"] = ""
                            print("Aucun bilan 21-24")
                    else:
                        target = int(val_annee)
                        df_batch.at[idx, "Total Actif"] = map_data.get(target, "")
                        print(f"Spec {target}: {map_data.get(target)}")

            except Exception as e:
                print(f"Erreur")
                df_batch.loc[df_batch[COL_SIREN] == siren, "Total Actif"] = ""

        # Sauvegarde du batch
        df_batch.to_excel(batch_filename, index=False)
        print(f"✓ Batch {i} sauvegardé dans {batch_filename}")

finally:
    driver.quit()

# 3. Fusion finale de tous les batchs
print("\n--- FUSION DES RÉSULTATS ---")
all_files = glob.glob("bdd_batch_*.xlsx")
if all_files:
    # On trie les fichiers par numéro pour garder l'ordre
    all_files.sort(key=lambda x: int(x.split('_')[-1].split('.')[0]))
    
    combined_df = pd.concat([pd.read_excel(f) for f in all_files], ignore_index=True)
    combined_df.to_excel(FINAL_OUTPUT, index=False)
    print(f"Succès ! {len(all_files)} batchs fusionnés dans {FINAL_OUTPUT}")
else:
    print("Aucun fichier batch trouvé pour la fusion.")

## Transformation de la base de données (un siren par ligne, pivoter les années dans les lignes en colonnes)

In [ ]:
import pandas as pd
import numpy as np

# --- 1. CONFIGURATION ---
FILE_INPUT = "bdd_complet_T_actif.xlsx"
FILE_OUTPUT = "bdd_final_propre.xlsx"
COL_ID = "SIREN"
COL_YEAR = "Année"

# Tes colonnes d'identité EXACTES
COL_IDENTITE = [
    "SIREN", 
    "Dénomination sociale de l'entreprise", 
    "Nom commercial de l'entreprise", 
    "Date societe a mission", 
    "Date de création", 
    "Code NAF - APE", 
    "Libelle APE", 
    "Date de vote de la mission par les actionnaires"
]

# Tes colonnes financières EXACTES
COL_FINANCE = [
    "chiffre d'affaires (€)", 
    "dettes financières (€)", 
    "fonds propres (€)", 
    "rentabilité sur fonds propres (%)", 
    "rentabilité économique (%)", 
    "roe calculé (%)", 
    "résultat d'exploitation (€)", 
    "résultat net (€)", 
    "Total Actif"
]

# --- 2. CHARGEMENT ---
print(f"Chargement de {FILE_INPUT}...")
df = pd.read_excel(FILE_INPUT, dtype={COL_ID: str})

# Vérification rapide pour toi dans Jupyter
print("Colonnes trouvées dans ton fichier :", df.columns.tolist())

# --- 3. EXTRACTION DE L'IDENTITÉ ---
# On s'assure de prendre toutes les colonnes d'identité demandées
# Si une colonne manque vraiment dans l'Excel, le script affichera un avertissement
cols_presentes = [c for c in COL_IDENTITE if c in df.columns]
cols_manquantes = [c for c in COL_IDENTITE if c not in df.columns]

if cols_manquantes:
    print(f"⚠️ Attention, colonnes introuvables : {cols_manquantes}")

df_identite = df[cols_presentes].drop_duplicates(subset=[COL_ID])

# --- 4. NETTOYAGE ET PIVOT ---
df_fin = df.copy()
df_fin[COL_YEAR] = pd.to_numeric(df_fin[COL_YEAR], errors='coerce')
df_fin = df_fin.dropna(subset=[COL_ID, COL_YEAR])
df_fin[COL_YEAR] = df_fin[COL_YEAR].astype(int)

# Filtrage 2021-2024
df_fin = df_fin[(df_fin[COL_YEAR] >= 2021) & (df_fin[COL_YEAR] <= 2024)]

# Conversion numérique (Texte -> NaN)
for col in COL_FINANCE:
    if col in df_fin.columns:
        df_fin[col] = pd.to_numeric(df_fin[col], errors='coerce')

# Pivotement
df_pivot = df_fin.pivot_table(index=COL_ID, columns=COL_YEAR, values=COL_FINANCE, aggfunc='first')
df_pivot = df_pivot.sort_index(axis=1, level=1) 
col_names_pivot = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
df_pivot.columns = col_names_pivot
df_pivot.reset_index(inplace=True)

# --- 5. FUSION ET CALCUL ROA ---
df_final = pd.merge(df_identite, df_pivot, on=COL_ID, how='left')

for annee in range(2021, 2025):
    col_net = f"résultat net (€)_{annee}"
    col_actif = f"Total Actif_{annee}"
    col_roa = f"ROA calculé (%)_{annee}"
    
    if col_net in df_final.columns and col_actif in df_final.columns:
        df_final[col_roa] = (df_final[col_net] / df_final[col_actif].replace(0, np.nan)) * 100
        df_final[col_roa] = df_final[col_roa].round(2)

# --- 6. NETTOYAGE DES LIGNES VIDES ---
cols_verif = [c for c in df_final.columns if any(str(y) in c for y in range(2021, 2025))]
df_final = df_final.dropna(subset=cols_verif, how='all')

# Sauvegarde
df_final.to_excel(FILE_OUTPUT, index=False)
print(f"\n✅ Terminé. Fichier '{FILE_OUTPUT}' prêt avec {len(df_final)} entreprises.")
df_final.head()